In [1]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil
import torch
import pandas as pd
import json

# Caminhos principais no RunPod
WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls'

DATASET_YAML_PATH_A = PROCESSED_DIR / 'dataset_path_A.yaml'
DATASET_YAML_PATH_B = PROCESSED_DIR / 'dataset_path_B.yaml'

# Path A: necessário porque o Path B usa o melhor detector treinado no Path A
RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A'

# Path B: treino do classificador
RUNS_PATH_B_DIR = WORKSPACE / 'runs' / 'path_B'
MLFLOW_DIR = Path('/root/mlflow')

# Resultados finais persistentes no volume (Alterado para não sobrescrever)
RESULTS_PATH_B_DIR = WORKSPACE / 'results_path_B'

# Scripts principais
TRAIN_PATH_B_SCRIPT = TRAIN_DIR / 'train_path_B.py'

for p in [RUNS_PATH_B_DIR, MLFLOW_DIR, RESULTS_PATH_B_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('TACO_DIR           =', TACO_DIR)
print('EXTERNAL_DIR       =', EXTERNAL_DIR)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_A=', DATASET_YAML_PATH_A)
print('DATASET_YAML_PATH_B=', DATASET_YAML_PATH_B)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('RUNS_PATH_B_DIR    =', RUNS_PATH_B_DIR)
print('MLFLOW_DIR         =', MLFLOW_DIR)
print('RESULTS_PATH_B_DIR =', RESULTS_PATH_B_DIR)
print('TRAIN_PATH_B_SCRIPT=', TRAIN_PATH_B_SCRIPT)

def run_cmd(cmd, cwd=WORKSPACE, env=None):
    """Roda comandos de forma previsível no notebook."""
    if isinstance(cmd, str):
        print('$', cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)

    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)

Python             = /workspace/.venv/bin/python
REPO_ROOT          = /workspace/TrashScan
TACO_DIR           = /workspace/TACO
EXTERNAL_DIR       = /workspace/external_datasets
PROCESSED_DIR      = /workspace/processed_5cls
DATASET_YAML_PATH_A= /workspace/processed_5cls/dataset_path_A.yaml
DATASET_YAML_PATH_B= /workspace/processed_5cls/dataset_path_B.yaml
RUNS_PATH_A_DIR    = /workspace/runs/path_A
RUNS_PATH_B_DIR    = /workspace/runs/path_B
MLFLOW_DIR         = /root/mlflow
RESULTS_PATH_B_DIR = /workspace/results_path_B
TRAIN_PATH_B_SCRIPT= /workspace/TrashScan/train/paths/train_path_B.py


In [2]:
required_paths = [
    TRAIN_PATH_B_SCRIPT,
    PROCESSED_DIR,
    RUNS_PATH_A_DIR,
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Arquivos/pastas ausentes:")
    for p in missing:
        print(" -", p)
else:
    print("Tudo certo.")

Tudo certo.


In [3]:
if torch.cuda.is_available():
    DEVICE = "0"
    gpu_name = torch.cuda.get_device_properties(0).name
else:
    DEVICE = "cpu"
    gpu_name = "cpu"

print("Device:", DEVICE)
print("GPU:", gpu_name)

EPOCHS = 50
BATCH = 32 
LR = 5e-5
PATIENCE = 10
DET_CONF = 0.25

CLASSIFIERS = [
    "resnet50", 
]

Device: 0
GPU: NVIDIA RTX A4500


In [4]:
PATH_A_RUN_DIRS = [
    WORKSPACE / "runs" / "path_A",
    WORKSPACE / "runs" / "path_A_5cls",
    WORKSPACE / "runs" / "path_A_refined_head",
]

def read_yolo_results(run_dir: Path):
    best_pt = run_dir / "weights" / "best.pt"
    results_csv = run_dir / "results.csv"
    metrics_json = run_dir / "metrics.json"

    if not best_pt.exists():
        return None

    row = {
        "group": run_dir.parent.name,
        "model": run_dir.name,
        "run_dir": run_dir,
        "best_pt": best_pt,
        "mAP50_95": None,
        "mAP50": None,
        "precision": None,
        "recall": None,
        "source": None,
    }

    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]

        map95_col = "metrics/mAP50-95(B)"
        map50_col = "metrics/mAP50(B)"
        precision_col = "metrics/precision(B)"
        recall_col = "metrics/recall(B)"

        if map95_col in df.columns:
            best_idx = df[map95_col].idxmax()
        elif map50_col in df.columns:
            best_idx = df[map50_col].idxmax()
        else:
            best_idx = df.index[-1]

        best = df.loc[best_idx]

        row["mAP50_95"] = float(best[map95_col]) if map95_col in df.columns else None
        row["mAP50"] = float(best[map50_col]) if map50_col in df.columns else None
        row["precision"] = float(best[precision_col]) if precision_col in df.columns else None
        row["recall"] = float(best[recall_col]) if recall_col in df.columns else None
        row["source"] = "results.csv"
        return row

    if metrics_json.exists():
        with open(metrics_json, "r") as f:
            m = json.load(f)

        row["mAP50_95"] = m.get("mAP50_95")
        row["mAP50"] = m.get("mAP50")
        row["precision"] = m.get("precision")
        row["recall"] = m.get("recall")
        row["source"] = "metrics.json"
        return row

    row["source"] = "weights_only"
    return row

records = []

for base_dir in PATH_A_RUN_DIRS:
    if not base_dir.exists():
        print(f"[warn] Pasta não encontrada: {base_dir}")
        continue

    for run_dir in sorted(base_dir.iterdir()):
        if not run_dir.is_dir():
            continue

        rec = read_yolo_results(run_dir)
        if rec is not None:
            records.append(rec)

df_detectors = pd.DataFrame(records)

if df_detectors.empty:
    raise FileNotFoundError(
        "Nenhum detector com weights/best.pt foi encontrado em: "
        + ", ".join(str(p) for p in PATH_A_RUN_DIRS)
    )

df_ranked = df_detectors.copy()
df_ranked["rank_score"] = df_ranked["mAP50_95"].fillna(df_ranked["mAP50"]).fillna(-1)

df_ranked = df_ranked.sort_values(
    by=["rank_score", "mAP50", "precision", "recall"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)

display_cols = [
    "group", "model", "mAP50_95", "mAP50", "precision", "recall", "source", "best_pt"
]

display(df_ranked[display_cols])

best_detector = df_ranked.iloc[0]
DETECTOR_WEIGHTS = Path(best_detector["best_pt"])

print("Melhor detector encontrado:")
print("Grupo :", best_detector["group"])
print("Modelo:", best_detector["model"])
print("mAP50-95:", best_detector["mAP50_95"])
print("mAP50:", best_detector["mAP50"])
print("Pesos:", DETECTOR_WEIGHTS)

if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(f"Detector não encontrado: {DETECTOR_WEIGHTS}")

,group,model,mAP50_95,mAP50,precision,recall,source,best_pt
0,path_A_5cls,yolov11m_o2o,0.49421,0.71055,0.79748,0.64537,results.csv,/workspace/runs/path_A_5cls/yolov11m_o2o/weigh...
1,path_A_5cls,yolov11m,0.45791,0.66802,0.76878,0.61979,results.csv,/workspace/runs/path_A_5cls/yolov11m/weights/b...
2,path_A_5cls,yolov8m,0.45466,0.67461,0.81395,0.59336,results.csv,/workspace/runs/path_A_5cls/yolov8m/weights/be...
3,path_A,yolov11m,0.44551,0.64990,0.70430,0.58665,results.csv,/workspace/runs/path_A/yolov11m/weights/best.pt
4,path_A,yolov11m_o2o,0.44040,0.64164,0.68838,0.59821,results.csv,/workspace/runs/path_A/yolov11m_o2o/weights/be...
5,path_A,yolov10m,0.39211,0.57796,0.63463,0.53399,results.csv,/workspace/runs/path_A/yolov10m/weights/best.pt
6,path_A,yolov9s,0.38005,0.57062,0.65521,0.51124,results.csv,/workspace/runs/path_A/yolov9s/weights/best.pt
7,path_A_refined_head,yolov10n,0.04614,0.11337,0.18439,0.16914,results.csv,/workspace/runs/path_A_refined_head/yolov10n/w...
8,path_A,yolov8m,NaN,NaN,NaN,NaN,weights_only,/workspace/runs/path_A/yolov8m/weights/best.pt


Melhor detector encontrado:
Grupo : path_A_5cls
Modelo: yolov11m_o2o
mAP50-95: 0.49421
mAP50: 0.71055
Pesos: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt


In [5]:
for split in ["train", "val", "test"]:
    path_b_dir = PROCESSED_DIR / split / "path_B"
    print(split, path_b_dir, "->", path_b_dir.exists())

    for sub in ["images", "labels", "crops"]:
        p = path_b_dir / sub
        print("  ", sub, "->", p.exists())

train /workspace/processed_5cls/train/path_B -> True
   images -> True
   labels -> True
   crops -> True
val /workspace/processed_5cls/val/path_B -> True
   images -> True
   labels -> True
   crops -> True
test /workspace/processed_5cls/test/path_B -> True
   images -> True
   labels -> True
   crops -> True


In [6]:
print("\nComando que será executado:")

print("\nParâmetros:")
print(f"Python executable: {sys.executable}")
print(f"Script: {TRAIN_PATH_B_SCRIPT}")
print(f"Detector weights: {DETECTOR_WEIGHTS}")
print(f"Crops dir: {PROCESSED_DIR}")
print(f"Output dir: {RUNS_PATH_B_DIR}")
print(f"Classifiers: {CLASSIFIERS}")
print(f"Epochs: {EPOCHS}")
print(f"Batch: {BATCH}")
print(f"Learning rate: {LR}")
print(f"Patience: {PATIENCE}")
print(f"Device: {DEVICE}")
print(f"Use YOLO crops: True")
print(f"Detection confidence: {DET_CONF}")
print(f"Crop cache dir: {RUNS_PATH_B_DIR / 'crop_cache'}")


Comando que será executado:

Parâmetros:
Python executable: /workspace/.venv/bin/python
Script: /workspace/TrashScan/train/paths/train_path_B.py
Detector weights: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
Crops dir: /workspace/processed_5cls
Output dir: /workspace/runs/path_B
Classifiers: ['resnet50']
Epochs: 50
Batch: 32
Learning rate: 5e-05
Patience: 10
Device: 0
Use YOLO crops: True
Detection confidence: 0.25
Crop cache dir: /workspace/runs/path_B/crop_cache


In [7]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--classifiers", *CLASSIFIERS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--lr", str(LR),
    "--patience", str(PATIENCE),
    "--device", str(DEVICE),
    "--use_yolo_crops",
    "--det_conf", str(DET_CONF),
    "--crop_cache_dir", str(RUNS_PATH_B_DIR / "crop_cache"),
])

$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B --classifiers resnet50 --epochs 50 --batch 32 --lr 5e-05 --patience 10 --device 0 --use_yolo_crops --det_conf 0.25 --crop_cache_dir /workspace/runs/path_B/crop_cache
Device : cuda:0
GPU    : NVIDIA RTX A4500  VRAM: 21.0GB
Class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

  Mode: YOLO on-the-fly crop extraction
  YOLO TTA+WBF: False
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


  [train] Loading YOLOCropDataset from cache: /workspace/runs/path_B/crop_cache/train
  [train] 11511 cached crops loaded
  [val] Loading YOLOCropDataset from cache: /workspace/runs/path_B/crop_cache/val
  [val] 2288 cached crops loaded
  [test] Loading YOLOCropDataset from cache: /workspace/runs/path_B/crop_cache/test
  [test] 2253 cached crops loaded

────────────────────────────────────────────────────────────
  Classifier : resnet50
────────────────────────────────────────────────────────────
  Built resnet50 (resnet50.a1_in1k)  pretrained=True  23.5M params


  Ep   1/50 | train loss=1.4187 acc=0.5596 | val loss=1.1129 acc=0.6010 f1=0.2662


  Ep   2/50 | train loss=1.2133 acc=0.6467 | val loss=0.9720 acc=0.6604 f1=0.3161


  Ep   3/50 | train loss=1.0967 acc=0.6796 | val loss=0.8418 acc=0.7054 f1=0.4289


...

  Ep  49/50 | train loss=0.2249 acc=0.9181 | val loss=0.3622 acc=0.9008 f1=0.8830


  Ep  50/50 | train loss=0.2370 acc=0.9158 | val loss=0.3657 acc=0.9052 f1=0.8849


  Test: 100%|██████████| 71/71 [00:05<00:00, 12.37it/s]



  [resnet50]  accuracy=0.9015  f1=0.8858  latency=6.91ms

  [resnet50]  best_epoch=50  val_acc=0.9052  test_acc=0.9015  f1=0.8858

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_l16_imagenet    0.9166     0.8934  0.9302 0.9088     22.4850      0.9690    0.9323    0.8562    0.9046    0.9686
resnet50            0.9015     0.8646  0.9130 0.8858      6.9080      0.9643    0.9073    0.8099    0.9391    0.9638


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B', '--classifiers', 'resnet50', '--epochs', '50', '--batch', '32', '--lr', '5e-05', '--patience', '10', '--device', '0', '--use_yolo_crops', '--det_conf', '0.25', '--crop_cache_dir', '/workspace/runs/path_B/crop_cache'], returncode=0)

In [8]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--classifiers", *CLASSIFIERS,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--lr", str(LR),
    "--patience", str(PATIENCE),
    "--device", str(DEVICE),
    "--use_yolo_crops",
    "--det_conf", str(DET_CONF),
    "--use_tta_wbf",
    "--tta_scales", "512", "640", "768",
    "--tta_flip",
    "--tta_wbf_iou", "0.55",
    "--tta_skip_box_thr", "0.001",
    "--crop_cache_dir", str(RUNS_PATH_B_DIR / "crop_cache_tta_wbf"),
])

$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B --classifiers resnet50 --epochs 50 --batch 32 --lr 5e-05 --patience 10 --device 0 --use_yolo_crops --det_conf 0.25 --use_tta_wbf --tta_scales 512 640 768 --tta_flip --tta_wbf_iou 0.55 --tta_skip_box_thr 0.001 --crop_cache_dir /workspace/runs/path_B/crop_cache_tta_wbf
Device : cuda:0
GPU    : NVIDIA RTX A4500  VRAM: 21.0GB
Class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

  Mode: YOLO on-the-fly crop extraction
  YOLO TTA+WBF: True
  TTA scales: [512, 640, 768]
  TTA flip: True
  TTA WBF IoU: 0.55
  TTA skip box thr: 0.001
  [train] Loading YOLOCropDataset from cache: /workspace/runs/path_B/crop_cache_tta_wbf/train
  [train] 11346 cached crops loaded
  [val] Loading YOLOCropDataset from cache: /workspace/runs/path_B/c

  Ep   1/50 | train loss=1.4054 acc=0.5767 | val loss=1.0984 acc=0.5905 f1=0.2610


  Ep   2/50 | train loss=1.2075 acc=0.6477 | val loss=0.9456 acc=0.6553 f1=0.2922


  Ep   3/50 | train loss=1.0883 acc=0.6836 | val loss=0.8379 acc=0.7067 f1=0.4444


  Ep   4/50 | train loss=0.9727 acc=0.7202 | val loss=0.7240 acc=0.7555 f1=0.5900


...

  Ep  49/50 | train loss=0.2233 acc=0.9222 | val loss=0.3556 acc=0.9015 f1=0.8834


  Ep  50/50 | train loss=0.2245 acc=0.9196 | val loss=0.3519 acc=0.9037 f1=0.8900


  Test: 100%|██████████| 70/70 [00:05<00:00, 13.29it/s]



  [resnet50]  accuracy=0.9063  f1=0.8879  latency=6.84ms

  [resnet50]  best_epoch=41  val_acc=0.9051  test_acc=0.9063  f1=0.8879

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_l16_imagenet    0.9166     0.8934  0.9302 0.9088     22.4850      0.9690    0.9323    0.8562    0.9046    0.9686
resnet50            0.9063     0.8637  0.9183 0.8879      6.8350      0.9646    0.9182    0.8335    0.9263    0.9663


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B', '--classifiers', 'resnet50', '--epochs', '50', '--batch', '32', '--lr', '5e-05', '--patience', '10', '--device', '0', '--use_yolo_crops', '--det_conf', '0.25', '--use_tta_wbf', '--tta_scales', '512', '640', '768', '--tta_flip', '--tta_wbf_iou', '0.55', '--tta_skip_box_thr', '0.001', '--crop_cache_dir', '/workspace/runs/path_B/crop_cache_tta_wbf'], returncode=0)

In [9]:
# Atualizado para o ResNet-50
run_dir = Path("/workspace/runs/path_B/resnet50")

print("best.pt:", (run_dir / "weights" / "best.pt").exists())
print("history.csv:", (run_dir / "history.csv").exists())
print("metrics.json:", (run_dir / "metrics.json").exists())

if (run_dir / "metrics.json").exists():
    print(json.loads((run_dir / "metrics.json").read_text()))

if (run_dir / "history.csv").exists():
    hist = pd.read_csv(run_dir / "history.csv")
    display(hist.tail())

best.pt: True
history.csv: True
metrics.json: True
{'classifier': 'resnet50', 'accuracy': 0.90635, 'precision': 0.86365, 'recall': 0.91827, 'f1': 0.88786, 'latency_ms': 6.835, 'AP_plastic': 0.96459, 'AP_paper': 0.91818, 'AP_metal': 0.83351, 'AP_glass': 0.92626, 'AP_other': 0.96628}


,epoch,train_loss,train_acc,val_loss,val_acc,val_f1,lr
45,46,0.22320,0.92006,0.35661,0.90328,0.88842,7.900000e-07
46,47,0.22410,0.91918,0.35546,0.90284,0.88252,4.400000e-07
47,48,0.22996,0.92041,0.35076,0.90195,0.88867,2.000000e-07
48,49,0.22333,0.92218,0.35563,0.90151,0.88340,5.000000e-08
49,50,0.22448,0.91962,0.35193,0.90373,0.89004,0.000000e+00


In [11]:
def run_cmd(cmd, cwd="/workspace", env=None, shell=False):
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    if shell:
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)
   
TRAIN_PATH_B_SCRIPT = Path("/workspace/TrashScan/train/paths/train_path_B.py")
RUNS_PATH_B_DIR = Path("/workspace/runs/path_B")
DETECTOR_WEIGHTS = Path("/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt")
PROCESSED_DIR = Path("/workspace/processed_5cls")

In [12]:
run_cmd([
    sys.executable, str(TRAIN_PATH_B_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--summarize",
])

$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_B.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --crops_dir /workspace/processed_5cls --output /workspace/runs/path_B --summarize

PATH B  —  CLASSIFIER BENCHMARK SUMMARY
                  accuracy  precision  recall     f1  latency_ms  AP_plastic  AP_paper  AP_metal  AP_glass  AP_other
classifier                                                                                                          
vit_l16_imagenet    0.9166     0.8934  0.9302 0.9088     22.4850      0.9690    0.9323    0.8562    0.9046    0.9686
resnet50            0.9063     0.8637  0.9183 0.8879      6.8350      0.9646    0.9182    0.8335    0.9263    0.9663


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_B.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--crops_dir', '/workspace/processed_5cls', '--output', '/workspace/runs/path_B', '--summarize'], returncode=0)

In [13]:
# Atualizado para o ResNet-50
RUN_DIR = Path("/workspace/runs/path_B/resnet50")

history_path = RUN_DIR / "history.csv"
metrics_path = RUN_DIR / "metrics.json"
best_weights_path = RUN_DIR / "weights" / "best.pt"
cm_path = RUN_DIR / "confusion_matrix_resnet50.png"

print("Run dir:", RUN_DIR)
print("best.pt existe?", best_weights_path.exists())
print("history.csv existe?", history_path.exists())
print("metrics.json existe?", metrics_path.exists())
print("confusion matrix existe?", cm_path.exists())

if not history_path.exists():
    raise FileNotFoundError(f"history.csv não encontrado: {history_path}")

hist = pd.read_csv(history_path)

best_val_acc_idx = hist["val_acc"].idxmax()
best_val_f1_idx = hist["val_f1"].idxmax()

best_val_acc_row = hist.loc[best_val_acc_idx]
best_val_f1_row = hist.loc[best_val_f1_idx]

summary = {
    "run_dir": str(RUN_DIR),
    "best_weights": str(best_weights_path),
    "best_weights_exists": best_weights_path.exists(),

    "epochs_trained": int(hist["epoch"].max()),

    "best_epoch_by_val_acc": int(best_val_acc_row["epoch"]),
    "best_val_acc": float(best_val_acc_row["val_acc"]),
    "best_val_acc_val_f1": float(best_val_acc_row["val_f1"]),
    "best_val_acc_train_acc": float(best_val_acc_row["train_acc"]),
    "best_val_acc_train_loss": float(best_val_acc_row["train_loss"]),
    "best_val_acc_val_loss": float(best_val_acc_row["val_loss"]),

    "best_epoch_by_val_f1": int(best_val_f1_row["epoch"]),
    "best_val_f1": float(best_val_f1_row["val_f1"]),
    "best_val_f1_val_acc": float(best_val_f1_row["val_acc"]),
}

if metrics_path.exists():
    with open(metrics_path, "r") as f:
        test_metrics = json.load(f)

    summary.update({
        "test_accuracy": test_metrics.get("accuracy"),
        "test_precision_macro": test_metrics.get("precision"),
        "test_recall_macro": test_metrics.get("recall"),
        "test_f1_macro": test_metrics.get("f1"),
        "latency_ms": test_metrics.get("latency_ms"),
        "AP_plastic": test_metrics.get("AP_plastic"),
        "AP_paper": test_metrics.get("AP_paper"),
        "AP_metal": test_metrics.get("AP_metal"),
        "AP_glass": test_metrics.get("AP_glass"),
        "AP_other": test_metrics.get("AP_other"),
    })
else:
    print("\n⚠️ metrics.json não foi encontrado. Talvez o erro no MLflow tenha ocorrido antes de salvar as métricas.")

summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
display(summary_df)

if metrics_path.exists():
    print("\nMétricas de teste:")
    display(pd.DataFrame([test_metrics]))

Run dir: /workspace/runs/path_B/resnet50
best.pt existe? True
history.csv existe? True
metrics.json existe? True
confusion matrix existe? True


,value
run_dir,/workspace/runs/path_B/resnet50
best_weights,/workspace/runs/path_B/resnet50/weights/best.pt
best_weights_exists,True
epochs_trained,50
best_epoch_by_val_acc,41
best_val_acc,0.90506
best_val_acc_val_f1,0.89209
best_val_acc_train_acc,0.92041
best_val_acc_train_loss,0.22742
best_val_acc_val_loss,0.35586



Métricas de teste:


,classifier,accuracy,precision,recall,f1,latency_ms,AP_plastic,AP_paper,AP_metal,AP_glass,AP_other
0,resnet50,0.90635,0.86365,0.91827,0.88786,6.835,0.96459,0.91818,0.83351,0.92626,0.96628


In [14]:
EVAL_PATH_B_COMBINED_SCRIPT = Path("/workspace/TrashScan/eval/evaluate_path_B_combined.py")

DETECTOR_WEIGHTS = Path("/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt")
CLASSIFIER_DIR = Path("/workspace/runs/path_B")
DATA_YAML = Path("/workspace/processed_5cls/dataset_path_B.yaml")

# A pasta de saída foi alterada para resnet50_5cls para não sobrescrever a vitl_5cls
OUTPUT_DIR = Path("/workspace/results_path_B/resnet50_5cls") 

cmd = [
    sys.executable, str(EVAL_PATH_B_COMBINED_SCRIPT),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--classifier_dir", str(CLASSIFIER_DIR),
    "--classifiers", "resnet50", # Atualizado para o ResNet-50
    "--data_yaml", str(DATA_YAML),
    "--output", str(OUTPUT_DIR),
    "--device", "0",
    "--imgsz", "640",
    "--det_conf", "0.001",
    "--det_iou", "0.6",
]

print("$", " ".join(shlex.quote(str(x)) for x in cmd))

subprocess.run(
    [str(x) for x in cmd],
    cwd="/workspace",
    check=True,
)

$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate_path_B_combined.py --detector_weights /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt --classifier_dir /workspace/runs/path_B --classifiers resnet50 --data_yaml /workspace/processed_5cls/dataset_path_B.yaml --output /workspace/results_path_B/resnet50_5cls --device 0 --imgsz 640 --det_conf 0.001 --det_iou 0.6
Device: cuda:0
Test set: /workspace/processed_5cls/test/path_B/images

Loading detector: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt

────────────────────────────────────────────────────────────
  Evaluating: resnet50
────────────────────────────────────────────────────────────
  Loaded resnet50: 23.5M params from best.pt

  Running combined inference on 1391 test images...
  TTA+WBF: False


  resnet50: 100%|██████████| 1391/1391 [02:21<00:00,  9.82it/s]



  [resnet50]  mAP50=0.6290  mAP50-95=0.4294  fps=9.8
    AP50 plastic : 0.6581  (n_gt=1313)
    AP50 paper   : 0.5947  (n_gt=280)
    AP50 metal   : 0.5364  (n_gt=265)
    AP50 glass   : 0.6678  (n_gt=88)
    AP50 other   : 0.6881  (n_gt=1263)
  Saved: /workspace/results_path_B/resnet50_5cls/individual/B_resnet50_combined.json

PATH B COMBINED — FINAL COMPARISON
Classifier                  mAP50  mAP50-95    FPS
──────────────────────────────────────────────────
  resnet50                 0.6290    0.4294    9.8

Summary saved: /workspace/results_path_B/resnet50_5cls/path_B_combined_summary.json


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate_path_B_combined.py', '--detector_weights', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt', '--classifier_dir', '/workspace/runs/path_B', '--classifiers', 'resnet50', '--data_yaml', '/workspace/processed_5cls/dataset_path_B.yaml', '--output', '/workspace/results_path_B/resnet50_5cls', '--device', '0', '--imgsz', '640', '--det_conf', '0.001', '--det_iou', '0.6'], returncode=0)